In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_predict, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score, confusion_matrix, make_scorer, roc_curve, auc
from sklearn.pipeline import Pipeline

# Regressão / Regression

## Com categoria / with label

Remove a coluna da categoria / remove class column

In [ ]:
# Load the dataset
try:
    df = pd.read_csv('NAME.csv', delimiter=',')
    print("CSV loaded successfully.")
    display(df.head())
except FileNotFoundError:
    print("Error: NAME.csv not found. Please make sure the file is in the correct directory.")
    # Removed !ls -F from print statement, can be run separately if needed.
    print("You can use `!ls -F` in a new cell to see available files if needed.")
    df = None

In [ ]:
if df is not None:
    # Remove the first column (categorical)
    df_clean = df.iloc[:, 1:]

    # Now:
    # Second column original → now becomes first column → target (y)
    y = df_clean.iloc[:, 0]

    # Remaining columns → features (X)
    X = df_clean.iloc[:, 1:]

    print(f"Shape of features (X): {X.shape}")
    print(f"Shape of target (y): {y.shape}")

    # Display first few rows
    print("Features (X) head:")
    display(X.head())

    print("Target (y) head:")
    display(y.head())

In [ ]:
if df is not None:
    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"Shape of scaled training features: {X_train_scaled.shape}")
    print(f"Shape of scaled testing features: {X_test_scaled.shape}")

In [ ]:
if y.dtype == 'object':
    y_numeric = y.astype(str).str.extract(r'(\d+\.?\d*)', expand=False).astype(float)
else:
    y_numeric = y.copy() # If already numeric, just make a copy

# Definir validação cruzada
cv = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scorer = make_scorer(lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)), greater_is_better=False)

# Set max_comp based on condition: if X.shape[1] > 40, max_comp = 40, otherwise max_comp = X.shape[1]
if X.shape[1] > 40:
    max_comp = 40
else:
    max_comp = X.shape[1]

# If X has no features, this will need further handling or raise an error.
if max_comp == 0:
    print("Error: X has no features. Cannot perform PLS regression.")
    rmse_cv = []
else:
    rmse_cv = []
    from sklearn.impute import SimpleImputer # Import SimpleImputer

    for n in range(1, max_comp + 1):
        pls = PLSRegression(n_components=n)
        # Use a pipeline to include scaling within cross-validation for robustness
        model_pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')), # Added Imputer to handle NaN values
            ('scaler', StandardScaler()),
            ('pls', pls)
        ])
        scores = cross_val_score(model_pipeline, X, y_numeric, cv=cv, scoring=rmse_scorer)
        rmse_cv.append(scores.mean())

    # Melhor número de componentes
    if rmse_cv:
        # Correctly find the best_n by applying argmin to positive RMSE values
        best_n = np.argmin([-score for score in rmse_cv]) + 1
        print("Melhor número de componentes:", best_n)

        # Plot the RMSE vs. Number of Components
        plt.figure(figsize=(10, 6))
        plt.plot(range(1, max_comp + 1), [-score for score in rmse_cv], marker='o') # Convert back to positive RMSE for plotting
        plt.axvline(x=best_n, color='r', linestyle='--', label=f'Optimal Components: {best_n}')
        plt.xlabel('Number of Components')
        plt.ylabel('Mean RMSE (Cross-Validation)')
        plt.title('RMSE vs. Number of Components for PLS Regression')
        plt.grid(True)
        plt.legend()
        plt.show()
    else:
        print("No RMSE values were calculated to determine the best number of components or plot.")

In [ ]:
if df is not None:
    # Initialize and train the PLS model
    # n_components is the number of latent variables to extract
    pls = PLSRegression(n_components=best_n)
    pls.fit(X_train_scaled, y_train)

    print("PLS Regression model trained successfully.")

In [ ]:
if df is not None:
    # Make predictions on the test set
    y_pred = pls.predict(X_test_scaled)

    # Convert y_test from string 'Xugml' to float X.0 for evaluation
    # This extracts the numerical part from the string, assuming it's at the beginning.
    y_test_numeric = y_test.astype(str).str.extract('(\\d+\\.?\\d*)', expand=False).astype(float)

    # Evaluate the model
    mse = mean_squared_error(y_test_numeric, y_pred)
    r2 = r2_score(y_test_numeric, y_pred)
    rmse = np.sqrt(mse)

    print(f"Mean Squared Error (MSE): {mse:.4f}")
    print(f"R-squared (R2): {r2:.4f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

    # Plotting actual vs. predicted values
    plt.figure(figsize=(10, 6))
    plt.scatter(y_test_numeric, y_pred, alpha=0.7)
    plt.plot([y_test_numeric.min(), y_test_numeric.max()], [y_test_numeric.min(), y_test_numeric.max()], 'k--', lw=2) # Diagonal line
    plt.xlabel('Actual Class Values')
    plt.ylabel('Predicted Class Values')
    plt.title('PLS Regression: Actual vs. Predicted Values')
    plt.grid(True)
    plt.show()

In [ ]:
if df is not None and 'pls' in locals():
    # Get PLS weights and Y scores
    x_weights = pls.x_weights_  # Shape: (n_features, n_components)
    y_scores = pls.y_scores_    # Shape: (n_samples, n_components)

    # Calculate the sum of squares of Y scores for each component
    ss_y_scores = np.sum(y_scores**2, axis=0) # Shape: (n_components,)

    # Calculate VIP scores
    # p is the number of features
    p = X.shape[1]
    # Calculate the numerator part: sum_a (w_ja^2 * SS_a)
    numerator = np.sum(x_weights**2 * ss_y_scores, axis=1)
    # Calculate the denominator part: sum_a (SS_a)
    denominator = np.sum(ss_y_scores)

    # Final VIP calculation
    vip_scores = np.sqrt(p * numerator / denominator)

    # Create a DataFrame for better visualization
    vip_df = pd.DataFrame({
        'Variable': X.columns,
        'VIP Score': vip_scores
    })

    # Sort by VIP Score in descending order
    vip_df = vip_df.sort_values(by='VIP Score', ascending=False).reset_index(drop=True)

    print("\nVariable Importance in Projection (VIP Scores):")
    display(vip_df)
else:
    print("PLS model not found or data not loaded. Please ensure previous cells ran successfully.")

## Sem categoria

In [ ]:
# Load the dataset
try:
    df = pd.read_csv('NAME.csv', sep=';')
    print("CSV loaded successfully.")
    display(df.head())
except FileNotFoundError:
    print("Error: NAME.csv not found. Please make sure the file is in the correct directory.")
    # Removed !ls -F from print statement, can be run separately if needed.
    print("You can use `!ls -F` in a new cell to see available files if needed.")
    df = None

In [ ]:
if df is not None:
    # Separate features (X) and target (y)
    # The first column is now assumed to be the class/target variable
    y = df.iloc[:, :1]  # First column
    X = df.iloc[:, 1:12] # Remaining columns are features (from second column onwards)

    print(f"Shape of features (X): {X.shape}")
    print(f"Shape of target (y): {y.shape}")

    # Display first few rows of X and y
    print("Features (X) head:")
    display(X.head())
    print("Target (y) head:")
    display(y.head())

In [ ]:
if df is not None:
    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"Shape of scaled training features: {X_train_scaled.shape}")
    print(f"Shape of scaled testing features: {X_test_scaled.shape}")

In [ ]:
if y.squeeze().dtype == 'object':
    y_numeric = y.astype(str).str.extract(r'(\d+\.?\d*)', expand=False).astype(float)
else:
    y_numeric = y.copy() # If already numeric, just make a copy

# Definir validação cruzada
cv = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scorer = make_scorer(lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)), greater_is_better=False)

# Set max_comp based on condition: if X.shape[1] > 40, max_comp = 40, otherwise max_comp = X.shape[1]
if X.shape[1] > 40:
    max_comp = 40
else:
    max_comp = X.shape[1]

# If X has no features, this will need further handling or raise an error.
if max_comp == 0:
    print("Error: X has no features. Cannot perform PLS regression.")
    rmse_cv = []
else:
    rmse_cv = []
    from sklearn.impute import SimpleImputer # Import SimpleImputer

    for n in range(1, max_comp + 1):
        pls = PLSRegression(n_components=n)
        # Use a pipeline to include scaling within cross-validation for robustness
        model_pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')), # Added Imputer to handle NaN values
            ('scaler', StandardScaler()),
            ('pls', pls)
        ])
        scores = cross_val_score(model_pipeline, X, y_numeric, cv=cv, scoring=rmse_scorer)
        rmse_cv.append(scores.mean())

    # Melhor número de componentes
    if rmse_cv:
        # Correctly find the best_n by applying argmin to positive RMSE values
        best_n = np.argmin([-score for score in rmse_cv]) + 1
        print("Melhor número de componentes:", best_n)

        # Plot the RMSE vs. Number of Components
        plt.figure(figsize=(10, 6))
        plt.plot(range(1, max_comp + 1), [-score for score in rmse_cv], marker='o') # Convert back to positive RMSE for plotting
        plt.axvline(x=best_n, color='r', linestyle='--', label=f'Optimal Components: {best_n}')
        plt.xlabel('Number of Components')
        plt.ylabel('Mean RMSE (Cross-Validation)')
        plt.title('RMSE vs. Number of Components for PLS Regression')
        plt.grid(True)
        plt.legend()
        plt.show()
    else:
        print("No RMSE values were calculated to determine the best number of components or plot.")

In [ ]:
if df is not None:
    # Initialize and train the PLS model
    # n_components is the number of latent variables to extract
    pls = PLSRegression(n_components=best_n)
    pls.fit(X_train_scaled, y_train)

    print("PLS Regression model trained successfully.")

In [ ]:
if df is not None:
    # Make predictions on the test set
    y_pred = pls.predict(X_test_scaled)

    # Evaluate the model
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mse)

    print(f"Mean Squared Error (MSE): {mse:.4f}")
    print(f"R-squared (R2): {r2:.4f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

    # Plotting actual vs. predicted values
    plt.figure(figsize=(10, 6))
    plt.scatter(y_test, y_pred, alpha=0.7)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2) # Diagonal line
    plt.xlabel('Actual Class Values')
    plt.ylabel('Predicted Class Values')
    plt.title('PLS Regression: Actual vs. Predicted Values')
    plt.grid(True)
    plt.show()

In [ ]:
if df is not None and 'pls' in locals():
    # Get PLS weights and Y scores
    x_weights = pls.x_weights_  # Shape: (n_features, n_components)
    y_scores = pls.y_scores_    # Shape: (n_samples, n_components)

    # Calculate the sum of squares of Y scores for each component
    ss_y_scores = np.sum(y_scores**2, axis=0) # Shape: (n_components,)

    # Calculate VIP scores
    # p is the number of features
    p = X.shape[1]
    # Calculate the numerator part: sum_a (w_ja^2 * SS_a)
    numerator = np.sum(x_weights**2 * ss_y_scores, axis=1)
    # Calculate the denominator part: sum_a (SS_a)
    denominator = np.sum(ss_y_scores)

    # Final VIP calculation
    vip_scores = np.sqrt(p * numerator / denominator)

    # Create a DataFrame for better visualization
    vip_df = pd.DataFrame({
        'Variable': X.columns,
        'VIP Score': vip_scores
    })

    # Sort by VIP Score in descending order
    vip_df = vip_df.sort_values(by='VIP Score', ascending=False).reset_index(drop=True)

    print("\nVariable Importance in Projection (VIP Scores):")
    display(vip_df)
else:
    print("PLS model not found or data not loaded. Please ensure previous cells ran successfully.")

# Discriminant Analysis

In [ ]:
# ==========================
# 1️⃣ Carregar dados / load data
# ==========================
dados = pd.read_csv('NAME.csv', sep=',')  # especifica o separador / specifies the separator
display(dados.head())

In [ ]:
# ==========================
# 2️⃣ Codificar variável alvo / Encode target variable
# ==========================

# Correctly define y_raw and X using the 'dados' DataFrame
y_raw = dados.iloc[:, 0]
X = dados.iloc[:, 1:]

label_enc = LabelEncoder()
y_encoded = label_enc.fit_transform(y_raw)

onehot = OneHotEncoder(sparse_output=False)
y_onehot = onehot.fit_transform(y_encoded.reshape(-1, 1))

print("Class:", label_enc.classes_)
print("Number of classes:", len(label_enc.classes_))

# ==========================
# 3️⃣ Separar treino e teste / Split test and train
# ==========================
X_train, X_test, y_train, y_test, y_encoded_train, y_encoded_test = train_test_split(
    X, y_onehot, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f"Shape of scaled training features: {X_train.shape}")
print(f"Shape of scaled testing features: {X_test.shape}")

print("\ny_raw head:")
display(y_raw.head())
print("\nX head:")
display(X.head())

In [ ]:
y_numeric = y_encoded

# Definir validação cruzada
cv = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scorer = make_scorer(lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)), greater_is_better=False)

# Set max_comp to be min of (number of samples in training fold) and number of features for PLS-DA
max_comp = min((X.shape[0] * (cv.n_splits - 1)) // cv.n_splits, X.shape[1])

# If X has no features or samples, this will need further handling or raise an error.
if max_comp == 0:
    print("Error: X has no features or samples. Cannot perform PLS regression.")
    rmse_cv = []
else:
    rmse_cv = []
    from sklearn.impute import SimpleImputer # Import SimpleImputer

    for n in range(1, max_comp + 1):
        pls = PLSRegression(n_components=n)
        # Use a pipeline to include scaling within cross-validation for robustness
        model_pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')), # Added Imputer to handle NaN values
            ('scaler', StandardScaler()),
            ('pls', pls)
        ])
        # Use y_encoded (which is y_numeric now) for cross-validation
        scores = cross_val_score(model_pipeline, X, y_numeric, cv=cv, scoring=rmse_scorer)
        rmse_cv.append(scores.mean())

    # Melhor número de componentes
    if rmse_cv:
        # Correctly find the best_n by applying argmin to positive RMSE values
        best_n = np.argmin([-score for score in rmse_cv]) + 1
        print("Melhor número de componentes:", best_n)

        # Get the RMSE value for the optimal number of components
        optimal_rmse = -rmse_cv[best_n - 1]

        # Plot the RMSE vs. Number of Components
        plt.figure(figsize=(10, 6))
        plt.plot(range(1, max_comp + 1), [-score for score in rmse_cv], marker='o') # Convert back to positive RMSE for plotting
        plt.axvline(x=best_n, color='r', linestyle='--', label=f'Optimal Components: {best_n} (RMSE: {optimal_rmse:.2f})')
        plt.xlabel('Number of Components')
        plt.ylabel('Mean RMSE (Cross-Validation)')
        plt.title('RMSE vs. Number of Components for PLS Regression')
        plt.grid(True)
        plt.legend()
        plt.show()
    else:
        print("No RMSE values were calculated to determine the best number of components or plot.")

In [ ]:
# ==========================
# 4️⃣ Padronização + PLS / Standardization + PLS
# ==========================
n_comp = best_n # number of labels/categories / número de variáveis/categorias

# Initialize and fit the scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Initialize and fit the PLS model
pls_da_model = PLSRegression(n_components=n_comp)
pls_da_model.fit(X_train_scaled, y_train)

print("PLS model trained successfully.")

In [ ]:
# ==========================
# 5️⃣ Predição / Prediction
# ==========================
# Scale the test features
X_test_scaled = scaler.transform(X_test)

y_pred_continuous = pls_da_model.predict(X_test_scaled)

y_pred_class = np.argmax(y_pred_continuous, axis=1)
y_test_class = np.argmax(y_test, axis=1)

# ==========================
# 6️⃣ Métricas / Metric
# ==========================
print("Accuracy:", accuracy_score(y_test_class, y_pred_class))
print("\nClassification report:\n")
report = classification_report(
    y_test_class,
    y_pred_class,
    target_names=label_enc.classes_.astype(str), # Convert class labels to strings
    output_dict=True # Output as dictionary to easily extract metrics
)
print(classification_report(
    y_test_class,
    y_pred_class,
    target_names=label_enc.classes_.astype(str) # Convert class labels to strings
))

mse = mean_squared_error(y_test, y_pred_continuous)

print("\nMSE:", mse)

# Calculate CCR (Class Classification Rate) - average of recall for each class
recalls = [report[class_name]['recall'] for class_name in label_enc.classes_.astype(str)]
ccr = np.mean(recalls)
print("CCR (Class Classification Rate):", ccr)

# ==========================
# 8️⃣ Matriz de Confusão / Confusion matrix
# ==========================
cm = confusion_matrix(y_test_class, y_pred_class)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=label_enc.classes_.astype(str), # Convert class labels to strings
            yticklabels=label_enc.classes_.astype(str),
            cmap="Blues")

plt.xlabel("Predict")
plt.ylabel("Real")
plt.title("Confusion Matrix - PLS-DA")
plt.tight_layout()
plt.show()

In [ ]:
# ==========================
# 1️⃣1️⃣ Curva ROC / ROC Curve
# ==========================

plt.figure(figsize=(10, 8))

# Store FPR, TPR for each class to calculate macro-average later
all_fpr = []
all_tpr = []
# Create a common set of FPRs for interpolation
mean_fpr = np.linspace(0, 1, 100)

for i, class_name in enumerate(label_enc.classes_):
    # y_test is already one-hot encoded, so we can directly use its columns
    y_true_class = y_test[:, i]
    y_score_class = y_pred_continuous[:, i]

    fpr, tpr, _ = roc_curve(y_true_class, y_score_class)
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f'ROC curve of class {class_name} (area = {roc_auc:.2f})', linewidth=2)

    # For macro-average, store interpolated TPR
    all_fpr.append(fpr)
    all_tpr.append(np.interp(mean_fpr, fpr, tpr)) # Interpolate TPR over mean_fpr

# Calculate Macro-average ROC curve and AUC
# Average all_tpr
macro_tpr = np.mean(all_tpr, axis=0)
macro_roc_auc = auc(mean_fpr, macro_tpr)

plt.plot(mean_fpr, macro_tpr,
         label=f'Macro-average ROC curve (area = {macro_roc_auc:.2f})',
         color='deeppink', linestyle=':', linewidth=4)

plt.plot([0, 1], [0, 1], 'k--', label='Random classifier (area = 0.50)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve for PLS-DA')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================
# 9️⃣ Score Plot (all Component Combinations)
# ==========================

X_train_scaled = scaler.transform(X_train)
scores = pls_da_model.transform(X_train_scaled)

# The number of optimal components found previously
num_components = best_n

# Loop through all unique combinations of components for plotting
for comp_x in range(num_components):
    for comp_y in range(comp_x + 1, num_components):
        plt.figure(figsize=(9, 6))

        for i, species in enumerate(label_enc.classes_):
            plt.scatter(scores[y_encoded_train == i, comp_x],
                        scores[y_encoded_train == i, comp_y],
                        label=species,
                        alpha=0.7)

        plt.xlabel(f"Component {comp_x + 1}")
        plt.ylabel(f"Component {comp_y + 1}")
        plt.title(f"Score Plot PLS-DA (Component {comp_x + 1} vs Component {comp_y + 1})")
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
        plt.tight_layout()
        plt.show()

In [ ]:
# ==========================
# 10. VIP Scores
# ==========================
if 'pls_da_model' in locals():
    # Get PLS weights and Y scores
    # x_weights_ are the weights for the X variables
    x_weights = pls_da_model.x_weights_  # Shape: (n_features, n_components)
    # y_scores_ are the scores for the Y variables (target)
    y_scores = pls_da_model.y_scores_    # Shape: (n_samples, n_components)

    # Calculate the sum of squares of Y scores for each component
    ss_y_scores = np.sum(y_scores**2, axis=0) # Shape: (n_components,)

    # Calculate VIP scores
    p = X.shape[1] # Number of features

    # Calculate the numerator part: sum_a (w_ja^2 * SS_a)
    numerator = np.sum(x_weights**2 * ss_y_scores, axis=1)
    # Calculate the denominator part: sum_a (SS_a)
    denominator = np.sum(ss_y_scores)

    # Final VIP calculation
    vip_scores = np.sqrt(p * numerator / denominator)

    # Create a DataFrame for better visualization
    vip_df = pd.DataFrame({
        'Variable': X.columns,
        'VIP Score': vip_scores
    })

    # Sort by VIP Score in descending order
    vip_df = vip_df.sort_values(by='VIP Score', ascending=False).reset_index(drop=True)

    print("\nVariable Importance in Projection (VIP Scores) for PLS-DA:")
    display(vip_df)

    # Save the VIP scores to a CSV file
    vip_df.to_csv('VIP.csv', index=False)
    print("VIP scores saved to VIP.csv")
else:
    print("PLS-DA model (pls_da_model) not found. Please ensure previous cells ran successfully.")

In [ ]:
if 'dados' in locals() and 'vip_df' in locals():
    # Prepare data for plotting
    # Combine spectral data (X) with original class labels (y_raw)
    df_plot_spectra = X.copy()
    df_plot_spectra['Class'] = y_raw

    # The spectral columns are all columns except 'Class'
    spectral_columns = df_plot_spectra.columns.drop('Class')

    # Calculate the mean spectrum for each class
    mean_spectra = df_plot_spectra.groupby('Class')[spectral_columns].mean()

    plt.figure(figsize=(14, 8))

    for i, (class_name, spectrum) in enumerate(mean_spectra.iterrows()):
        plt.plot(spectrum.index.astype(float), spectrum.values, label=f'Class: {class_name}')

    # Get the top 10 VIP variables
    top_10_vip = vip_df.head(10)

    # Highlight the top 10 VIP variables
    for idx, row in top_10_vip.iterrows():
        plt.axvline(x=float(row['Variable']), color='red', linestyle='--', linewidth=1,
                    label=f"VIP {idx+1}: {row['Variable']} ({row['VIP Score']:.2f})",
                    alpha=0.7)

    plt.xlabel('Wavenumber (cm$^{-1}$)')
    plt.ylabel('Intensity (a.u.)')
    plt.title('Mean Spectrum per Class with Top 10 VIP Variables Highlighted')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("Required DataFrames ('dados' or 'vip_df') not found. Please ensure previous cells ran successfully.")